<a href="https://colab.research.google.com/github/hebawl/starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**My lane:** Structured Content Archetype Clustering (from `w02_ml_task_framing.ipynb`) — clustering content items into recurring performance archetypes (champions, rising stars, hidden gems, stale visible pages, etc.) using visibility, engagement, freshness, and depth signals. Unsupervised, so there is no target label — the "prediction" is cluster membership, discovered after the fact.

## 0. Connect to the warehouse

*One-time setup: DuckDB reads the Hugging Face release directly. Token goes through `getpass` / Colab Secrets — never pasted in a cell, this repo is public.*

In [3]:
%pip -q install duckdb huggingface_hub


In [4]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_march':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Sanity check against the numbers the docs promise (dim_content: 519,606 | fact_daily: 78,835,655)
for name in ['dim_clients', 'dim_content', 'fact_daily']:
    n = con.sql(f"SELECT COUNT(*) FROM {TABLES[name]}").fetchone()[0]
    print(f'{name:16} {n:>12,} rows')


dim_clients               104 rows
dim_content           519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily         78,835,655 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [6]:
# CONTRACT — part 1: unit of analysis + time window
#
# One row = ONE CONTENT ITEM (content_hash_id), as it behaved during a single
# calendar month, built by aggregating its daily rows in
# fact_content_daily_performance UP to that content item's grain, then joined
# to its static metadata in dim_content.
#
# Time window: I develop on month = 2026-03 (mid-panel, per the warehouse
# warning — the _sample table and the final month=2026-06 partition are the
# sealed test window and are never used to build label/feature logic here).
# I read the March partition directly
# (fact_content_daily_performance/month=2026-03/*.parquet) instead of
# scanning all 78.8M rows and filtering — partition pruning does the
# filtering for free.
print('Unit of analysis: one row = one content item, aggregated over March 2026 (month=2026-03).')


Unit of analysis: one row = one content item, aggregated over March 2026 (month=2026-03).


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
# CONTRACT — part 2: field classification
#
# FEATURE (knowable from March's own observed activity or from static
# content metadata — safe to cluster on):
#   - gsc_impressions (fact_daily, summed over March)         -> visibility volume
#   - gsc_avg_position (fact_daily, averaged over March)      -> visibility quality
#   - an engagement column discovered on fact_daily below      -> engagement
#   - content_created_at (dim_content) -> content_age_days     -> freshness
#   - word_count (dim_content, if present)                     -> content depth
#
# LABEL / PROXY: none for the core lane — clustering is unsupervised, there
# is no target column. The only place a "label" appears is the throwaway
# demo in section 5, built purely to stage the leakage trap, and it is never
# used for the clustering itself.
#
# CONTEXT (for grouping/joining/reading only, never a feature):
#   - content_hash_id  -> the grain key, joins fact_daily to dim_content
#   - client_hash_id    -> client grouping, per-client history checks
#   - report_date       -> defines the March window, not a feature
#   - ga4_data_available -> filter flag, not a feature
#
# EXCLUDED:
#   - gsc_clicks / ctr -> excluded from the cluster features on purpose:
#     CTR is a ratio of two things already represented (impressions, and
#     position as a proxy for rank), and the lane brief's four archetype
#     dimensions are visibility / freshness / engagement / depth — adding
#     CTR would double-count visibility rather than add a new dimension.
#   - keyword_hash_id / url_hash_id (dim_content) -> scrambled join keys
#     only, never model input (per skills/flyrank-data).
#   - any product-decision columns (health_score, priority_score, action
#     flags) -> not shipped in this release at all, so nothing to exclude
#     in practice, but noting it because it's the reason the release looks
#     the way it does.
print('Fields sorted into feature / label / context / excluded above.')


Fields sorted into feature / label / context / excluded above.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

First, look at the real column names instead of guessing them — `DESCRIBE` on an empty `SELECT *` costs nothing and stops the rest of the notebook from being built on invented column names.

In [8]:
# Discover real column names — don't hardcode guesses
fact_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_march']} LIMIT 0").df()['column_name'].tolist()
dim_cols  = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 0").df()['column_name'].tolist()
client_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']} LIMIT 0").df()['column_name'].tolist()

print('fact_content_daily_performance columns:', fact_cols)
print()
print('dim_content columns:', dim_cols)
print()
print('dim_clients columns:', client_cols)

def find_col(cols, *keywords, exclude=()):
    """Return the first column whose name contains all keywords and none of the excluded substrings."""
    for c in cols:
        lc = c.lower()
        if all(k in lc for k in keywords) and not any(x in lc for x in exclude):
            return c
    return None

ENGAGEMENT_COL = find_col(fact_cols, 'session', exclude=('ai',)) or find_col(fact_cols, 'engag')
WORDCOUNT_COL  = find_col(dim_cols, 'word')
CREATED_COL    = find_col(dim_cols, 'created') or find_col(dim_cols, 'age')
print()
print('Engagement column discovered:', ENGAGEMENT_COL)
print('Word count column discovered:', WORDCOUNT_COL)
print('Content created/age column discovered:', CREATED_COL)


fact_content_daily_performance columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized

**Query 1 — grain.** One row per content item, once I aggregate the March daily rows down to that grain. Empty result = grain holds.

In [9]:
# Grain check: after aggregating to content_hash_id, no id should repeat.
grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS n
    FROM (
        SELECT DISTINCT content_hash_id, client_hash_id
        FROM {TABLES['fact_daily_march']}
    )
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f'Content items appearing under more than one client_hash_id: {len(grain_check)} (0 = grain holds, a content item lives with exactly one client)')
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items appearing under more than one client_hash_id: 0 (0 = grain holds, a content item lives with exactly one client)


,content_hash_id,n


**Query 2 — row count + date span.** My slice's size, and whether the dates it actually covers match what I claimed (all of March 2026, not partial).

In [10]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_march']}
""").df()
print('March 2026 partition, my slice:')
span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 partition, my slice:


,n_rows,n_content_items,min_date,max_date
0,9841378,331437,2026-03-01,2026-03-31


**Query 3 — availability.** GA4 columns are zero-filled (not truly missing) before a client's `ga4_data_start`. Filtering on `ga4_data_available IS TRUE` shows how many rows are real engagement data versus filler.

In [11]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_real_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_real_ga4
    FROM {TABLES['fact_daily_march']}
""").df()
print('Availability check — rows that survive an IS TRUE filter on ga4_data_available:')
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability check — rows that survive an IS TRUE filter on ga4_data_available:


,total_rows,rows_with_real_ga4,pct_with_real_ga4
0,9841378,413966.0,4.2


## 4. Five features

*Build a small feature frame for my lane from the same March month. One line per feature: knowable at the decision moment because…*

For clustering there's no future to leak from — I'm describing pages as of the end of March, not predicting what they'll do next. "Knowable at the decision moment" here means: built only from March's own observed activity, or from static content metadata, never from anything about April onward.

In [12]:
fact_march_src = TABLES['fact_daily_march']
dim_content_src = TABLES['dim_content']

features = con.sql(f"""
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id)                                   AS client_hash_id,
        SUM(f.gsc_impressions)                                        AS total_impressions_30d,
        AVG(NULLIF(f.gsc_avg_position, 0))                            AS avg_position_30d,
        SUM(f.{ENGAGEMENT_COL})                                       AS engagement_30d
    FROM {fact_march_src} f
    GROUP BY f.content_hash_id
    HAVING SUM(f.gsc_impressions) > 0
""").df()

meta = con.sql(f"""
    SELECT
        content_hash_id,
        DATE_DIFF('day', {CREATED_COL}, DATE '2026-03-31') AS content_age_days,
        {WORDCOUNT_COL} AS word_count
    FROM {dim_content_src}
""").df()

feature_frame = features.merge(meta, on='content_hash_id', how='left')
print(f'{len(feature_frame):,} content items x {feature_frame.shape[1]} columns')
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 content items x 7 columns


,content_hash_id,client_hash_id,total_impressions_30d,avg_position_30d,engagement_30d,content_age_days,word_count
0,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.209549,1.0,396,keyword_bb608ec2b5123615
1,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,3.307255,0.0,396,keyword_d687914aaaa4be3e
2,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.724039,3.0,396,keyword_d870b3a452bf1eeb
3,content_a7da352b73b02668,client_73cda7b4e4f265ea,4944.0,7.244844,2.0,396,keyword_ed9d2014b30f85a9
4,content_f39be42b42a4e8f6,client_73cda7b4e4f265ea,42.0,23.314103,7.0,396,keyword_70c20a4fd380bab8


In [13]:
# Five features, each with an "available when?" line:
#
# 1. total_impressions_30d  -- visibility volume.
#    Knowable at the decision moment because it is summed only over March's
#    own report_date rows -- no day outside the window contributes to it.
#
# 2. avg_position_30d       -- visibility quality (lower = better rank).
#    Knowable at the decision moment because it's an average of March's own
#    daily gsc_avg_position values; NULLIF(...,0) keeps "no position data"
#    days out of the average instead of pulling it toward zero.
#
# 3. engagement_30d (from the discovered ENGAGEMENT_COL) -- engagement.
#    Knowable at the decision moment because it's summed from March's own
#    daily rows, same as impressions -- it never touches April+ data.
#
# 4. content_age_days       -- freshness/tenure.
#    Knowable at the decision moment because it's the gap between the page's
#    creation date (fixed, in the past) and the snapshot date (2026-03-31) --
#    pure arithmetic on a timestamp that was already true before March started.
#
# 5. word_count             -- content depth.
#    Knowable at the decision moment because it's static content metadata --
#    it doesn't change based on how the page performs.
print('Five features defined; see comments above for the available-when line per feature.')


Five features defined; see comments above for the available-when line per feature.


## 5. The trap: a deliberate leak

*Add ONE label-derived column on purpose, watch the quick score jump toward perfect, then delete it and keep the honest number — the leakage lesson from notebook 02, performed on real warehouse data.*

Clustering itself has no label to leak. So this section stages a small, separate, throwaway demo — same shape as notebook 03's decline model — purely to reproduce the leakage trap on real warehouse data, then discards it. It never feeds the actual clustering work above.

In [14]:
# Demo target (throwaway, NOT part of the clustering lane):
# split March in two — days 1-14 (pre) build the features, days 16-31 (post)
# define the label. "Did this page's impressions drop >20% from pre to post?"
demo = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-14' THEN gsc_impressions ELSE 0 END) AS imp_pre,
        AVG(CASE WHEN report_date <= DATE '2026-03-14' THEN NULLIF(gsc_avg_position,0) END) AS pos_pre,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_post
    FROM {TABLES['fact_daily_march']}
    GROUP BY content_hash_id
    HAVING imp_pre >= 50
""").df()

demo['is_declining'] = (demo['imp_post'] < 0.8 * demo['imp_pre']).astype(int)
print(f'{len(demo):,} content items with enough pre-period volume | decline rate: {demo["is_declining"].mean():.3f}')


90,593 content items with enough pre-period volume | decline rate: 0.258


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def quick_score(frame, feature_cols):
    d = frame.dropna(subset=feature_cols + ['is_declining'])
    X, y = d[feature_cols], d['is_declining']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

# Honest version: only pre-period features, nothing from the label window.
honest_auc = quick_score(demo, ['imp_pre', 'pos_pre'])
print(f'Honest AUC (pre-period features only): {honest_auc:.3f}')


Honest AUC (pre-period features only): 0.511


In [16]:
# The trap: add imp_post -- a column literally used to COMPUTE the label --
# as if it were just another feature.
leaked_auc = quick_score(demo, ['imp_pre', 'pos_pre', 'imp_post'])
print(f'Leaked AUC (imp_post included, the label-derived column): {leaked_auc:.3f}')
print()
print(f'Honest: {honest_auc:.3f}  ->  Leaked: {leaked_auc:.3f}  (jumps toward 1.0 because the model is just reading the label back)')


Leaked AUC (imp_post included, the label-derived column): 1.000

Honest: 0.511  ->  Leaked: 1.000  (jumps toward 1.0 because the model is just reading the label back)


In [17]:
# Delete the leak, keep the honest number.
# imp_post is dropped -- it is never used again in this notebook, and it was
# never part of the actual clustering feature_frame from section 4 either.
FINAL_DEMO_AUC = honest_auc
print(f'Kept: honest AUC = {FINAL_DEMO_AUC:.3f}. imp_post removed from the feature set.')


Kept: honest AUC = 0.511. imp_post removed from the feature set.


## 6. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
# One named limitation: this slice is a single calendar month (March 2026)
# from an unbalanced panel -- per-client history depth ranges from 3 to 17
# months (dim_clients.gsc_data_start), and a third of clients have little or
# no usable search/analytics history at all. A one-month cut can't tell me
# whether a page's archetype is stable across seasons or just what March
# looked like for it -- a page that reads as "hidden gem" here might be
# "stale" in a different month. It also can't separate a client that simply
# started tracking in March (few real days of data) from one with a genuine
# March slowdown, which is exactly what the ga4_data_available /
# gsc_data_start checks in section 3 exist to catch -- worth re-running this
# contract against a second month before trusting any cluster as durable.
print('Limitation recorded above.')


Limitation recorded above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.